# Triagegeist — End-to-End Pipeline
### Emergency Severity Index Prediction from Structured Vitals + Clinical NLP

**Full pipeline — run top to bottom on Kaggle. All artefacts saved to `/kaggle/working/`.**

| Phase | Scope |
|---|---|
| Phase 1 | EDA + baseline LightGBM (structured features only) |
| Phase 2 | Feature engineering + Bio_ClinicalBERT NLP branch + full LightGBM |
| Phase 3 | Optuna HPO + XGBoost ensemble + bias analysis + SHAP explainability |

> **GitHub:** https://github.com/sayan1506/Triagegeist

## Section 1 — Setup & Data Loading

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

WORKING_DIR = '/kaggle/working/'
DATA_DIR    = '/kaggle/input/competitions/triagegeist/'

train_raw    = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_raw     = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
complaints   = pd.read_csv(os.path.join(DATA_DIR, 'chief_complaints.csv'))
patient_hist = pd.read_csv(os.path.join(DATA_DIR, 'patient_history.csv'))

complaints_clean = complaints[['patient_id', 'chief_complaint_raw']].rename(
    columns={'chief_complaint_raw': 'complaint_text'})

train = train_raw.merge(patient_hist, on='patient_id', how='left')
train = train.merge(complaints_clean, on='patient_id', how='left')
test  = test_raw.merge(patient_hist,  on='patient_id', how='left')
test  = test.merge(complaints_clean,  on='patient_id', how='left')

cols_to_drop = ['chief_complaint_system_x', 'chief_complaint_system_y', 'chief_complaint_system']
train = train.drop(columns=[c for c in cols_to_drop if c in train.columns])
test  = test.drop(columns=[c for c in cols_to_drop if c in test.columns])

TARGET    = 'triage_acuity'
DROP_COLS = ['patient_id', 'site_id', 'triage_nurse_id',
             'disposition', 'ed_los_hours', 'complaint_text', TARGET]

train_text = train['complaint_text'].copy()
test_text  = test['complaint_text'].copy()

feature_cols = [c for c in train.columns if c not in DROP_COLS]
X_train_raw  = train[feature_cols].copy()
y_train      = train[TARGET]
X_test_raw   = test[feature_cols].copy()

print(f'Train: {train.shape}  |  Test: {test.shape}')
print(f'Target distribution:\n{y_train.value_counts().sort_index()}')

## Section 2 — Exploratory Data Analysis

In [ ]:
# 2.1 Class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
y_train.value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('ESI Class Distribution (Count)')
axes[0].set_xlabel('Triage Acuity (ESI)')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=10)
(y_train.value_counts().sort_index() / len(y_train) * 100).plot(
    kind='bar', ax=axes[1], color='coral', edgecolor='black')
axes[1].set_title('ESI Class Distribution (%)')
axes[1].set_xlabel('Triage Acuity (ESI)')
axes[1].set_ylabel('Percentage (%)')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig(WORKING_DIR + 'eda_class_distribution.png')
plt.show()
print('Class counts:\n', y_train.value_counts().sort_index())

In [ ]:
# 2.2 Missingness heatmap
missing = X_train_raw.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
missing.plot(kind='bar', ax=axes[0], color='tomato', edgecolor='black')
axes[0].set_title('Missing Value Counts')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
train_with_target = X_train_raw.copy()
train_with_target['triage_acuity'] = y_train.values
miss_by_esi = train_with_target.groupby('triage_acuity')[missing.index.tolist()].apply(
    lambda x: x.isnull().mean() * 100)
sns.heatmap(miss_by_esi.T, annot=True, fmt='.1f', cmap='Reds', ax=axes[1],
            cbar_kws={'label': 'Missing %'})
axes[1].set_title('Missingness % by ESI Group')
plt.tight_layout()
plt.savefig(WORKING_DIR + 'eda_missingness.png')
plt.show()
print('Missing value summary:\n', missing)

In [ ]:
# 2.3 Vital signs boxplots by ESI level
vital_cols = ['heart_rate', 'spo2', 'respiratory_rate', 'temperature_c',
              'systolic_bp', 'diastolic_bp', 'gcs_total', 'pain_score',
              'news2_score', 'shock_index']
train_plot = X_train_raw[vital_cols].copy()
train_plot['triage_acuity'] = y_train.values
fig, axes = plt.subplots(2, 5, figsize=(22, 10))
axes = axes.flatten()
for i, col in enumerate(vital_cols):
    sns.boxplot(data=train_plot, x='triage_acuity', y=col,
                ax=axes[i], palette='coolwarm')
    axes[i].set_title(f'{col} vs ESI')
plt.suptitle('Vital Signs by ESI Level', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(WORKING_DIR + 'eda_vitals_boxplots.png')
plt.show()

In [ ]:
# 2.4 Categorical features vs ESI
cat_cols_eda = ['arrival_mode', 'sex', 'age_group', 'insurance_type', 'shift']
train_cat = X_train_raw[cat_cols_eda].copy()
train_cat['triage_acuity'] = y_train.values
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()
for i, col in enumerate(cat_cols_eda):
    ct = pd.crosstab(train_cat[col], train_cat['triage_acuity'], normalize='index') * 100
    ct.plot(kind='bar', ax=axes[i], colormap='tab10', edgecolor='black')
    axes[i].set_title(f'{col} vs ESI (%)')
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=30, ha='right')
    axes[i].legend(title='ESI', bbox_to_anchor=(1, 1))
axes[-1].set_visible(False)
plt.suptitle('Categorical Features vs ESI Level', fontsize=14)
plt.tight_layout()
plt.savefig(WORKING_DIR + 'eda_categorical.png')
plt.show()

In [ ]:
# 2.5 Age distribution by ESI
train_age = X_train_raw[['age']].copy()
train_age['triage_acuity'] = y_train.values
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for esi in sorted(train_age['triage_acuity'].unique()):
    train_age[train_age['triage_acuity'] == esi]['age'].plot(
        kind='kde', ax=axes[0], label=f'ESI {esi}')
axes[0].set_title('Age Distribution by ESI Level (KDE)')
axes[0].legend(title='ESI')
sns.boxplot(data=train_age, x='triage_acuity', y='age', ax=axes[1], palette='viridis')
axes[1].set_title('Age vs ESI Level')
plt.tight_layout()
plt.savefig(WORKING_DIR + 'eda_age.png')
plt.show()
print('Median age by ESI:\n', train_age.groupby('triage_acuity')['age'].median())

In [ ]:
# 2.6 Comorbidity prevalence heatmap
hx_cols = [c for c in X_train_raw.columns if c.startswith('hx_')]
train_hx = X_train_raw[hx_cols].copy()
train_hx['triage_acuity'] = y_train.values
hx_by_esi = train_hx.groupby('triage_acuity')[hx_cols].mean() * 100
fig, ax = plt.subplots(figsize=(18, 7))
sns.heatmap(hx_by_esi.T, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
            cbar_kws={'label': 'Prevalence %'})
ax.set_title('Comorbidity Prevalence (%) by ESI Level')
plt.tight_layout()
plt.savefig(WORKING_DIR + 'eda_comorbidities.png')
plt.show()

In [ ]:
# 2.7 Temporal patterns
train_time = X_train_raw[['arrival_hour', 'arrival_day', 'arrival_month', 'shift']].copy()
train_time['triage_acuity'] = y_train.values
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
train_time.groupby(['arrival_hour', 'triage_acuity']).size().unstack().plot(
    ax=axes[0], colormap='tab10')
axes[0].set_title('Arrivals by Hour and ESI')
train_time.groupby(['arrival_month', 'triage_acuity']).size().unstack().plot(
    ax=axes[1], colormap='tab10')
axes[1].set_title('Arrivals by Month and ESI')
pd.crosstab(train_time['shift'], train_time['triage_acuity'],
            normalize='index').mul(100).plot(
    kind='bar', ax=axes[2], colormap='tab10', edgecolor='black')
axes[2].set_title('ESI Distribution by Shift (%)')
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.savefig(WORKING_DIR + 'eda_temporal.png')
plt.show()

In [ ]:
# 2.8 Chief complaint word frequencies by ESI
train_text_df = pd.DataFrame({'complaint_text': train_text.values,
                               'triage_acuity': y_train.values})
stopwords = {'the', 'a', 'and', 'of', 'to', 'in', 'with', 'no', 'for', 'on'}
fig, axes = plt.subplots(1, 5, figsize=(25, 6))
for esi in range(1, 6):
    subset = train_text_df[train_text_df['triage_acuity'] == esi]['complaint_text']
    words  = ' '.join(subset.dropna()).lower().split()
    words  = [w for w in words if w not in stopwords and len(w) > 2]
    top    = Counter(words).most_common(15)
    wl, ct = zip(*top)
    axes[esi-1].barh(wl[::-1], ct[::-1], color='steelblue')
    axes[esi-1].set_title(f'ESI {esi} — Top Words')
plt.suptitle('Top Chief Complaint Words by ESI Level', fontsize=14)
plt.tight_layout()
plt.savefig(WORKING_DIR + 'eda_complaints_wordfreq.png')
plt.show()

In [ ]:
# 2.9 Correlation heatmap + target correlations
numeric_cols = X_train_raw.select_dtypes(include=[np.number]).columns.tolist()
train_corr = X_train_raw[numeric_cols].copy()
train_corr['triage_acuity'] = y_train.values
corr_matrix = train_corr.corr()
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', center=0, ax=axes[0], annot=False)
axes[0].set_title('Feature Correlation Matrix')
target_corr = corr_matrix['triage_acuity'].drop('triage_acuity').sort_values()
target_corr.plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='black')
axes[1].set_title('Feature Correlation with triage_acuity')
axes[1].axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig(WORKING_DIR + 'eda_correlation.png')
plt.show()
print('Top 10 features correlated with target:\n',
      target_corr.abs().sort_values(ascending=False).head(10))

In [ ]:
# 2.10 Outlier / range check
vital_check = ['heart_rate', 'spo2', 'respiratory_rate', 'temperature_c',
               'systolic_bp', 'diastolic_bp', 'gcs_total', 'pain_score',
               'age', 'bmi', 'weight_kg', 'height_cm']
print('=== Outlier / Range Check ===\n')
for col in vital_check:
    if col in X_train_raw.columns:
        print(f'{col}: min={X_train_raw[col].min():.1f}, '
              f'max={X_train_raw[col].max():.1f}, '
              f'mean={X_train_raw[col].mean():.1f}, '
              f'null={X_train_raw[col].isnull().sum()}')
print('\nPain score -1 count:', (X_train_raw['pain_score'] == -1).sum())

## Section 3 — Phase 1: Baseline Model
Structured features only, no NLP. Establishes the score to beat.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, cohen_kappa_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import lightgbm as lgb

X_bl      = X_train_raw.copy()
X_bl_test = X_test_raw.copy()

# Pain score sentinel fix
X_bl['pain_missing']      = (X_bl['pain_score'] == -1).astype(int)
X_bl_test['pain_missing'] = (X_bl_test['pain_score'] == -1).astype(int)
X_bl['pain_score']        = X_bl['pain_score'].replace(-1, float('nan'))
X_bl_test['pain_score']   = X_bl_test['pain_score'].replace(-1, float('nan'))

# Global median imputation
for col in X_bl.columns:
    if X_bl[col].isnull().sum() > 0:
        m = X_bl[col].median()
        X_bl[col]      = X_bl[col].fillna(m)
        X_bl_test[col] = X_bl_test[col].fillna(m)

# Label encode
cat_cols_bl = X_bl.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols_bl:
    le = LabelEncoder()
    combined = pd.concat([X_bl[col], X_bl_test[col]]).astype(str)
    le.fit(combined)
    X_bl[col]      = le.transform(X_bl[col].astype(str))
    X_bl_test[col] = le.transform(X_bl_test[col].astype(str))

print('Nulls after imputation:', X_bl.isnull().sum().sum())
print('Baseline feature count:', X_bl.shape[1])

In [ ]:
y = y_train.values - 1  # 0-indexed for LightGBM

classes           = np.unique(y)
weights           = compute_class_weight('balanced', classes=classes, y=y)
class_weight_dict = dict(zip(classes, weights))
sample_weights    = np.array([class_weight_dict[yi] for yi in y])

baseline_params = {
    'objective': 'multiclass', 'num_class': 5, 'metric': 'multi_logloss',
    'num_leaves': 63, 'learning_rate': 0.05, 'feature_fraction': 0.8,
    'bagging_fraction': 0.8, 'bagging_freq': 5, 'min_child_samples': 20,
    'verbose': -1, 'random_state': 42, 'n_jobs': -1
}

skf       = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_bl    = np.zeros((len(X_bl), 5))
test_bl   = np.zeros((len(X_bl_test), 5))
bl_f1s, bl_kappas = [], []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_bl, y)):
    print(f'\n--- Baseline Fold {fold+1}/5 ---')
    X_tr, X_val = X_bl.iloc[tr_idx], X_bl.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    sw_tr       = sample_weights[tr_idx]
    dtrain = lgb.Dataset(X_tr, label=y_tr, weight=sw_tr)
    dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)
    mdl = lgb.train(baseline_params, dtrain, num_boost_round=1000,
                    valid_sets=[dval],
                    callbacks=[lgb.early_stopping(50, verbose=False),
                               lgb.log_evaluation(100)])
    vp = mdl.predict(X_val)
    oof_bl[val_idx] = vp
    test_bl += mdl.predict(X_bl_test) / 5
    f1    = f1_score(y_val, np.argmax(vp, 1), average='weighted')
    kappa = cohen_kappa_score(y_val, np.argmax(vp, 1), weights='quadratic')
    bl_f1s.append(f1); bl_kappas.append(kappa)
    print(f'Fold {fold+1} — F1: {f1:.4f} | QWK: {kappa:.4f}')

oof_f1    = f1_score(y, np.argmax(oof_bl, 1), average='weighted')
oof_kappa = cohen_kappa_score(y, np.argmax(oof_bl, 1), weights='quadratic')
oof_acc   = (np.argmax(oof_bl, 1) == y).mean()

print('\n========== BASELINE RESULTS ==========')
print(f'OOF Weighted F1 : {oof_f1:.4f}')
print(f'OOF QWK         : {oof_kappa:.4f}')
print(f'OOF Accuracy    : {oof_acc:.4f}')
print('======================================')
print(classification_report(y, np.argmax(oof_bl, 1),
      target_names=['ESI 1','ESI 2','ESI 3','ESI 4','ESI 5']))

np.save(WORKING_DIR + 'oof_baseline.npy', oof_bl)
np.save(WORKING_DIR + 'test_baseline.npy', test_bl)
print('Baseline predictions saved.')

## Section 4 — Phase 2: Structured Feature Engineering

In [ ]:
# 4.1 Missingness flags (before imputation)
missing_flag_cols = ['systolic_bp', 'diastolic_bp', 'mean_arterial_pressure',
                     'pulse_pressure', 'shock_index', 'respiratory_rate', 'temperature_c']
for col in missing_flag_cols:
    if col in train.columns:
        train[f'{col}_missing'] = train[col].isnull().astype(int)
        test[f'{col}_missing']  = test[col].isnull().astype(int)

train['pain_missing'] = (train['pain_score'] == -1).astype(int)
test['pain_missing']  = (test['pain_score']  == -1).astype(int)
train['pain_score']   = train['pain_score'].replace(-1, float('nan'))
test['pain_score']    = test['pain_score'].replace(-1, float('nan'))

flag_cols = [c for c in train.columns if c.endswith('_missing')]
print('Missingness flags added:')
print(train[flag_cols].sum())

In [ ]:
# 4.2 ESI-group aware median imputation
impute_cols = ['systolic_bp', 'diastolic_bp', 'mean_arterial_pressure',
               'pulse_pressure', 'shock_index', 'respiratory_rate',
               'temperature_c', 'pain_score']
esi_medians = train.groupby(TARGET)[impute_cols].median()
print('ESI-group medians:\n', esi_medians.round(2))

for col in impute_cols:
    if col in train.columns:
        for esi_level in train[TARGET].unique():
            mask = (train[TARGET] == esi_level) & (train[col].isnull())
            train.loc[mask, col] = esi_medians.loc[esi_level, col]

for col in impute_cols:
    if col in test.columns:
        test[col] = test[col].fillna(train[col].median())

print('Nulls after imputation (train):', train[impute_cols].isnull().sum().sum())
print('Nulls after imputation (test): ', test[impute_cols].isnull().sum().sum())

In [ ]:
# 4.3 Cyclical time encoding
for col in ['arrival_hour', 'arrival_day', 'arrival_month']:
    train[col] = pd.to_numeric(train[col], errors='coerce')
    test[col]  = pd.to_numeric(test[col],  errors='coerce')

train['arrival_hour_sin']  = np.sin(2*np.pi*train['arrival_hour']/24)
train['arrival_hour_cos']  = np.cos(2*np.pi*train['arrival_hour']/24)
test['arrival_hour_sin']   = np.sin(2*np.pi*test['arrival_hour']/24)
test['arrival_hour_cos']   = np.cos(2*np.pi*test['arrival_hour']/24)
train['arrival_day_sin']   = np.sin(2*np.pi*train['arrival_day']/7)
train['arrival_day_cos']   = np.cos(2*np.pi*train['arrival_day']/7)
test['arrival_day_sin']    = np.sin(2*np.pi*test['arrival_day']/7)
test['arrival_day_cos']    = np.cos(2*np.pi*test['arrival_day']/7)
train['arrival_month_sin'] = np.sin(2*np.pi*train['arrival_month']/12)
train['arrival_month_cos'] = np.cos(2*np.pi*train['arrival_month']/12)
test['arrival_month_sin']  = np.sin(2*np.pi*test['arrival_month']/12)
test['arrival_month_cos']  = np.cos(2*np.pi*test['arrival_month']/12)

train = train.drop(columns=['arrival_hour', 'arrival_day', 'arrival_month'])
test  = test.drop(columns=['arrival_hour', 'arrival_day', 'arrival_month'])
print('Cyclical time features added.')

In [ ]:
# 4.4 Label encode categoricals on combined train+test
feature_cols = [c for c in train.columns if c not in DROP_COLS]
X_train_eng  = train[feature_cols].copy()
X_test_eng   = test[feature_cols].copy()
y            = train[TARGET].values - 1

cat_cols = X_train_eng.select_dtypes(include=['object']).columns.tolist()
print('Encoding:', cat_cols)
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([X_train_eng[col], X_test_eng[col]]).astype(str)
    le.fit(combined)
    X_train_eng[col] = le.transform(X_train_eng[col].astype(str))
    X_test_eng[col]  = le.transform(X_test_eng[col].astype(str))
    le_dict[col] = le

print('X_train_eng shape:', X_train_eng.shape)
print('Remaining nulls:',   X_train_eng.isnull().sum().sum())

## Section 5 — Phase 2: NLP Branch

In [ ]:
# 5.1 Keyword risk flags
import re

keyword_patterns = {
    'kw_chest_pain':          r'chest pain',
    'kw_shortness_of_breath': r'short(ness)? of breath|sob|dyspn',
    'kw_syncope':             r'syncope|fainted|passed out',
    'kw_altered_ms':          r'altered mental status|confusion|disoriented|ams',
    'kw_stroke':              r'stroke|facial droop|arm weakness|slurred speech',
    'kw_unresponsive':        r'unresponsive|unconscious|not responding',
    'kw_severe_pain':         r'severe pain|10/10|worst pain',
    'kw_diaphoresis':         r'diaphoresis|diaphoretic|sweating profusely',
    'kw_hypotension':         r'hypotension|hypotensive|low blood pressure',
    'kw_trauma':              r'trauma|mvc|motor vehicle|fall from|gunshot|stab',
    'kw_cardiac_arrest':      r'cardiac arrest|no pulse|pulseless',
    'kw_seizure':             r'seizure|convulsion|fitting',
    'kw_severe':              r'\bsevere\b',
    'kw_acute':               r'\bacute\b',
    'kw_massive':             r'\bmassive\b',
}

def add_keyword_flags(text_series):
    return pd.DataFrame({
        name: text_series.str.contains(pat, case=False, regex=True, na=False).astype(int)
        for name, pat in keyword_patterns.items()
    })

train_keywords = add_keyword_flags(train_text)
test_keywords  = add_keyword_flags(test_text)
print('Keyword flags shape:', train_keywords.shape)
print(train_keywords.sum().sort_values(ascending=False))

In [ ]:
# 5.2 TF-IDF features (500 features, unigrams + bigrams)
from sklearn.feature_extraction.text import TfidfVectorizer

train_text_filled = train_text.fillna('').astype(str)
test_text_filled  = test_text.fillna('').astype(str)

tfidf = TfidfVectorizer(max_features=500, ngram_range=(1, 2),
                        min_df=5, max_df=0.95, sublinear_tf=True)
tfidf.fit(train_text_filled)

train_tfidf_arr = tfidf.transform(train_text_filled).toarray()
test_tfidf_arr  = tfidf.transform(test_text_filled).toarray()

tfidf_cols     = [f'tfidf_{i}' for i in range(train_tfidf_arr.shape[1])]
train_tfidf_df = pd.DataFrame(train_tfidf_arr, columns=tfidf_cols)
test_tfidf_df  = pd.DataFrame(test_tfidf_arr,  columns=tfidf_cols)
print('TF-IDF shape:', train_tfidf_df.shape)

In [ ]:
# 5.3 Bio_ClinicalBERT embeddings
# Note: runs on CPU — GPU P100 has a kernel image mismatch with this PyTorch build
import torch
from transformers import AutoTokenizer, AutoModel

device     = torch.device('cpu')
model_path = 'emilyalsentzer/Bio_ClinicalBERT'

tokenizer  = AutoTokenizer.from_pretrained(model_path)
bert_model = AutoModel.from_pretrained(model_path).to(device)
bert_model.eval()
print(f'Bio_ClinicalBERT loaded on {device}.')

def get_bert_embeddings(texts, batch_size=32):
    all_emb = []
    texts = texts.fillna('').astype(str).tolist()
    for i in range(0, len(texts), batch_size):
        batch   = texts[i:i+batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True,
                            max_length=128, return_tensors='pt')
        encoded = {k: v.to(device) for k, v in encoded.items()}
        with torch.no_grad():
            out = bert_model(**encoded)
        all_emb.append(out.last_hidden_state[:, 0, :].cpu().numpy())
        if (i // batch_size) % 10 == 0:
            print(f'  {i+len(batch)}/{len(texts)} processed')
    return np.vstack(all_emb)

print('Generating train BERT embeddings...')
train_bert = get_bert_embeddings(train_text)
print('Generating test BERT embeddings...')
test_bert  = get_bert_embeddings(test_text)

np.save(WORKING_DIR + 'train_bert.npy', train_bert)
np.save(WORKING_DIR + 'test_bert.npy',  test_bert)
print(f'BERT embeddings saved. Shape: {train_bert.shape}')

In [ ]:
# 5.4 PCA compression 768 → 64 dimensions
from sklearn.decomposition import PCA

train_bert = np.load(WORKING_DIR + 'train_bert.npy')
test_bert  = np.load(WORKING_DIR + 'test_bert.npy')

pca_check = PCA(n_components=100)
pca_check.fit(train_bert)
cumvar = np.cumsum(pca_check.explained_variance_ratio_)

plt.figure(figsize=(10, 4))
plt.plot(range(1, 101), cumvar, marker='o', markersize=3)
plt.axhline(y=0.90, color='r', linestyle='--', label='90% variance')
plt.axhline(y=0.95, color='g', linestyle='--', label='95% variance')
plt.xlabel('Components'); plt.ylabel('Cumulative Variance')
plt.title('BERT PCA Scree Plot'); plt.legend()
plt.tight_layout()
plt.savefig(WORKING_DIR + 'pca_scree.png')
plt.show()
print(f'Components for 90% variance: {np.argmax(cumvar >= 0.90)+1}')
print(f'Components for 95% variance: {np.argmax(cumvar >= 0.95)+1}')

N_COMPONENTS   = 64
pca            = PCA(n_components=N_COMPONENTS, random_state=42)
pca.fit(train_bert)
train_bert_pca = pca.transform(train_bert)
test_bert_pca  = pca.transform(test_bert)

bert_cols     = [f'bert_{i}' for i in range(N_COMPONENTS)]
train_bert_df = pd.DataFrame(train_bert_pca, columns=bert_cols)
test_bert_df  = pd.DataFrame(test_bert_pca,  columns=bert_cols)
print(f'BERT PCA shape: {train_bert_df.shape} | Variance retained: {pca.explained_variance_ratio_.sum():.3f}')

## Section 6 — Phase 2: Feature Fusion + Full LightGBM

In [ ]:
# 6.1 Concatenate all feature branches
X_train_eng    = X_train_eng.reset_index(drop=True)
X_test_eng     = X_test_eng.reset_index(drop=True)
train_bert_df  = train_bert_df.reset_index(drop=True)
test_bert_df   = test_bert_df.reset_index(drop=True)
train_tfidf_df = train_tfidf_df.reset_index(drop=True)
test_tfidf_df  = test_tfidf_df.reset_index(drop=True)
train_keywords = train_keywords.reset_index(drop=True)
test_keywords  = test_keywords.reset_index(drop=True)

X_train_full = pd.concat([X_train_eng, train_bert_df, train_tfidf_df, train_keywords], axis=1)
X_test_full  = pd.concat([X_test_eng,  test_bert_df,  test_tfidf_df,  test_keywords],  axis=1)

print('=' * 50)
print(f'X_train_full : {X_train_full.shape}')
print(f'X_test_full  : {X_test_full.shape}')
print(f'  Structured : {X_train_eng.shape[1]}')
print(f'  BERT PCA   : {train_bert_df.shape[1]}')
print(f'  TF-IDF     : {train_tfidf_df.shape[1]}')
print(f'  Keywords   : {train_keywords.shape[1]}')
print(f'  TOTAL      : {X_train_full.shape[1]}')
print(f'Null check   : {X_train_full.isnull().sum().sum()}')
print('=' * 50)

In [ ]:
# 6.2 Full LightGBM with all features — 5-fold CV
classes           = np.unique(y)
weights           = compute_class_weight('balanced', classes=classes, y=y)
class_weight_dict = dict(zip(classes, weights))
sample_weights    = np.array([class_weight_dict[yi] for yi in y])

phase2_params = {
    'objective': 'multiclass', 'num_class': 5, 'metric': 'multi_logloss',
    'num_leaves': 63, 'learning_rate': 0.05, 'feature_fraction': 0.8,
    'bagging_fraction': 0.8, 'bagging_freq': 5, 'min_child_samples': 20,
    'verbose': -1, 'random_state': 42, 'n_jobs': -1
}

skf           = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds_v2  = np.zeros((len(X_train_full), 5))
test_preds_v2 = np.zeros((len(X_test_full), 5))
fold_f1s_v2, fold_kappas_v2 = [], []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_full, y)):
    print(f'\n--- Fold {fold+1}/5 ---')
    X_tr, X_val = X_train_full.iloc[tr_idx], X_train_full.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    sw_tr       = sample_weights[tr_idx]
    dtrain = lgb.Dataset(X_tr, label=y_tr, weight=sw_tr)
    dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)
    mdl = lgb.train(phase2_params, dtrain, num_boost_round=1000,
                    valid_sets=[dval],
                    callbacks=[lgb.early_stopping(50, verbose=False),
                               lgb.log_evaluation(100)])
    vp = mdl.predict(X_val)
    oof_preds_v2[val_idx] = vp
    test_preds_v2 += mdl.predict(X_test_full) / 5
    f1    = f1_score(y_val, np.argmax(vp, 1), average='weighted')
    kappa = cohen_kappa_score(y_val, np.argmax(vp, 1), weights='quadratic')
    fold_f1s_v2.append(f1); fold_kappas_v2.append(kappa)
    print(f'Fold {fold+1} — F1: {f1:.4f} | QWK: {kappa:.4f}')

oof_f1_v2    = f1_score(y, np.argmax(oof_preds_v2, 1), average='weighted')
oof_kappa_v2 = cohen_kappa_score(y, np.argmax(oof_preds_v2, 1), weights='quadratic')
print('\n========== PHASE 2 RESULTS ==========')
print(f'OOF Weighted F1 : {oof_f1_v2:.4f}')
print(f'OOF QWK         : {oof_kappa_v2:.4f}')
print(f'vs Baseline F1  : {oof_f1:.4f} | Delta: {oof_f1_v2 - oof_f1:+.4f}')
print('=====================================')

In [ ]:
# 6.3 Save Phase 2 outputs
np.save(WORKING_DIR + 'oof_phase2.npy',  oof_preds_v2)
np.save(WORKING_DIR + 'test_phase2.npy', test_preds_v2)

X_train_full.to_parquet(WORKING_DIR + 'X_train_full.parquet')
X_test_full.to_parquet(WORKING_DIR  + 'X_test_full.parquet')

train_meta_cols = ['patient_id', 'sex', 'age_group', 'arrival_mode', 'insurance_type', TARGET]
train[train_meta_cols].to_csv(WORKING_DIR + 'train_meta.csv', index=False)

summary_p2 = {
    'baseline':   {'f1': round(oof_f1, 4),    'qwk': round(oof_kappa, 4)},
    'full_nlp':   {'f1': round(oof_f1_v2, 4), 'qwk': round(oof_kappa_v2, 4)},
    'nlp_uplift': {'f1': round(oof_f1_v2 - oof_f1, 4)},
    'features':   X_train_full.shape[1]
}
with open(WORKING_DIR + 'phase1_phase2_summary.json', 'w') as f:
    json.dump(summary_p2, f, indent=2)

print('Phase 1+2 outputs saved.')
print(json.dumps(summary_p2, indent=2))

## Section 7 — Phase 3: Optuna Hyperparameter Optimisation

In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'objective':         'multiclass',
        'num_class':         5,
        'metric':            'multi_logloss',
        'verbosity':         -1,
        'random_state':      42,
        'n_jobs':            -1,
        'num_leaves':        trial.suggest_int('num_leaves', 31, 255),
        'learning_rate':     trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'feature_fraction':  trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction':  trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq':      trial.suggest_int('bagging_freq', 1, 10),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0.0, 10.0),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0.0, 10.0),
        'min_split_gain':    trial.suggest_float('min_split_gain', 0.0, 1.0),
    }
    skf_opt     = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    fold_scores = []
    for tr_idx, val_idx in skf_opt.split(X_train_full, y):
        X_tr  = X_train_full.iloc[tr_idx]
        X_val = X_train_full.iloc[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]
        sw_tr = sample_weights[tr_idx]
        dtrain = lgb.Dataset(X_tr, label=y_tr, weight=sw_tr)
        dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)
        model  = lgb.train(params, dtrain, num_boost_round=500,
                           valid_sets=[dval],
                           callbacks=[lgb.early_stopping(30, verbose=False),
                                      lgb.log_evaluation(-1)])
        val_proba = model.predict(X_val)
        fold_scores.append(f1_score(y_val, np.argmax(val_proba, 1), average='weighted'))
    return np.mean(fold_scores)

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=10)
)

print('Starting Optuna HPO — 20 trials...')
study.optimize(objective, n_trials=20, show_progress_bar=True)

print('\n========== OPTUNA RESULTS ==========')
print(f'Best F1 (3-fold CV): {study.best_value:.4f}')
print('Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')
print('=====================================')

best_params = study.best_params.copy()
best_params.update({'objective': 'multiclass', 'num_class': 5,
                    'metric': 'multi_logloss', 'verbosity': -1,
                    'random_state': 42, 'n_jobs': -1})

## Section 8 — Phase 3: Tuned LightGBM (5-Fold CV)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_lgbm    = np.zeros((len(X_train_full), 5))
test_lgbm   = np.zeros((len(X_test_full), 5))
lgbm_models = []
fold_f1s, fold_kappas = [], []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_full, y)):
    print(f'\n--- Fold {fold+1}/5 ---')
    X_tr  = X_train_full.iloc[tr_idx]
    X_val = X_train_full.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    sw_tr = sample_weights[tr_idx]
    dtrain = lgb.Dataset(X_tr, label=y_tr, weight=sw_tr)
    dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)
    model  = lgb.train(best_params, dtrain, num_boost_round=1000,
                       valid_sets=[dval],
                       callbacks=[lgb.early_stopping(50, verbose=False),
                                  lgb.log_evaluation(100)])
    val_proba = model.predict(X_val)
    oof_lgbm[val_idx] = val_proba
    test_lgbm += model.predict(X_test_full) / 5
    f1    = f1_score(y_val, np.argmax(val_proba, 1), average='weighted')
    kappa = cohen_kappa_score(y_val, np.argmax(val_proba, 1), weights='quadratic')
    fold_f1s.append(f1); fold_kappas.append(kappa)
    lgbm_models.append(model)
    print(f'Fold {fold+1} — F1: {f1:.4f} | QWK: {kappa:.4f}')

lgbm_f1    = f1_score(y, np.argmax(oof_lgbm, 1), average='weighted')
lgbm_kappa = cohen_kappa_score(y, np.argmax(oof_lgbm, 1), weights='quadratic')

print('\n========== TUNED LGBM RESULTS ==========')
print(f'OOF Weighted F1 : {lgbm_f1:.4f}')
print(f'OOF QWK         : {lgbm_kappa:.4f}')
print(f'vs Phase 2 F1   : {oof_f1_v2:.4f} | Delta: {lgbm_f1 - oof_f1_v2:+.4f}')
print('=========================================')
print('\nPer-class report:')
print(classification_report(y, np.argmax(oof_lgbm, 1),
      target_names=['ESI 1','ESI 2','ESI 3','ESI 4','ESI 5']))

np.save(WORKING_DIR + 'oof_lgbm_tuned.npy', oof_lgbm)
np.save(WORKING_DIR + 'test_lgbm_tuned.npy', test_lgbm)
print('Saved tuned LightGBM predictions.')

## Section 9 — Phase 3: XGBoost
CPU mode (`tree_method='hist'`) — avoids CUDA kernel mismatch on Kaggle P100.

In [ ]:
import xgboost as xgb

xgb_params = {
    'objective':        'multi:softprob',
    'num_class':        5,
    'eval_metric':      'mlogloss',
    'learning_rate':    0.05,
    'max_depth':        6,
    'min_child_weight': 5,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'reg_alpha':        0.1,
    'reg_lambda':       1.0,
    'random_state':     42,
    'tree_method':      'hist',
    'verbosity':        0
}

skf         = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_xgb     = np.zeros((len(X_train_full), 5))
test_xgb    = np.zeros((len(X_test_full), 5))
fold_f1s_xgb = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_full, y)):
    print(f'\n--- XGBoost Fold {fold+1}/5 ---')
    X_tr  = X_train_full.iloc[tr_idx].values
    X_val = X_train_full.iloc[val_idx].values
    y_tr, y_val = y[tr_idx], y[val_idx]
    sw_tr = sample_weights[tr_idx]
    dtrain    = xgb.DMatrix(X_tr,  label=y_tr,  weight=sw_tr)
    dval      = xgb.DMatrix(X_val, label=y_val)
    dtest     = xgb.DMatrix(X_test_full.values)
    model_xgb = xgb.train(xgb_params, dtrain, num_boost_round=1000,
                           evals=[(dval, 'val')],
                           early_stopping_rounds=50,
                           verbose_eval=100)
    val_proba = model_xgb.predict(dval).reshape(-1, 5)
    oof_xgb[val_idx] = val_proba
    test_xgb += model_xgb.predict(dtest).reshape(-1, 5) / 5
    f1 = f1_score(y_val, np.argmax(val_proba, 1), average='weighted')
    fold_f1s_xgb.append(f1)
    print(f'Fold {fold+1} — F1: {f1:.4f}')

xgb_f1    = f1_score(y, np.argmax(oof_xgb, 1), average='weighted')
xgb_kappa = cohen_kappa_score(y, np.argmax(oof_xgb, 1), weights='quadratic')
print('\n========== XGBOOST RESULTS ==========')
print(f'OOF Weighted F1 : {xgb_f1:.4f}')
print(f'OOF QWK         : {xgb_kappa:.4f}')
print('=====================================')

np.save(WORKING_DIR + 'oof_xgb.npy',  oof_xgb)
np.save(WORKING_DIR + 'test_xgb.npy', test_xgb)
print('Saved XGBoost predictions.')

## Section 10 — Phase 3: Ensemble Blending
Grid search over LGBM + XGBoost blend weights. MLP excluded due to training instability on unscaled mixed-range features.

In [ ]:
from sklearn.metrics import f1_score, cohen_kappa_score
from scipy.stats import rankdata

oof_lgbm  = np.load(WORKING_DIR + 'oof_lgbm_tuned.npy')
oof_xgb   = np.load(WORKING_DIR + 'oof_xgb.npy')
test_lgbm = np.load(WORKING_DIR + 'test_lgbm_tuned.npy')
test_xgb  = np.load(WORKING_DIR + 'test_xgb.npy')

print('Individual model OOF scores:')
print(f'  LightGBM : F1={f1_score(y, np.argmax(oof_lgbm, 1), average="weighted"):.4f}')
print(f'  XGBoost  : F1={f1_score(y, np.argmax(oof_xgb,  1), average="weighted"):.4f}')

# Grid search over 2-model blend weights
print('\nSearching best blend weights (LGBM + XGBoost)...')
best_blend_f1 = 0
best_w        = (0.5, 0.5)
results       = []

for w_lgbm in np.arange(0.2, 0.9, 0.05):
    w_xgb = round(1.0 - w_lgbm, 2)
    blend = w_lgbm * oof_lgbm + w_xgb * oof_xgb
    f1    = f1_score(y, np.argmax(blend, 1), average='weighted')
    results.append((round(w_lgbm, 2), round(w_xgb, 2), round(f1, 4)))
    if f1 > best_blend_f1:
        best_blend_f1 = f1
        best_w = (w_lgbm, w_xgb)

results.sort(key=lambda x: x[2], reverse=True)
print(f'{"LGBM":>6} {"XGB":>6} {"F1":>8}')
for r in results[:5]:
    print(f'{r[0]:>6.2f} {r[1]:>6.2f} {r[2]:>8.4f}')

print(f'\nBest weights: LGBM={best_w[0]:.2f}, XGB={best_w[1]:.2f}')
print(f'Best blend F1: {best_blend_f1:.4f}')

final_test_preds = best_w[0]*test_lgbm + best_w[1]*test_xgb
final_oof_preds  = best_w[0]*oof_lgbm  + best_w[1]*oof_xgb
final_oof_labels = np.argmax(final_oof_preds, axis=1)

final_f1    = f1_score(y, final_oof_labels, average='weighted')
final_kappa = cohen_kappa_score(y, final_oof_labels, weights='quadratic')

print('\n========== FINAL ENSEMBLE ==========')
print(f'Method    : LGBM+XGB soft blend')
print(f'OOF F1    : {final_f1:.4f}')
print(f'OOF QWK   : {final_kappa:.4f}')
print('=====================================')

np.save(WORKING_DIR + 'final_test_preds.npy', final_test_preds)
np.save(WORKING_DIR + 'final_oof_preds.npy',  final_oof_preds)
print('Final ensemble predictions saved.')

## Section 11 — Phase 3: Bias Analysis
Undertriage and overtriage rates by sex, age group, and arrival mode.

In [ ]:
# Load train metadata for subgroup columns
train_meta = pd.read_csv(WORKING_DIR + 'train_meta.csv')

bias_df = pd.DataFrame({
    'true_esi':      y + 1,
    'pred_esi':      final_oof_labels + 1,
    'sex':           train_meta['sex'].values,
    'age_group':     train_meta['age_group'].values,
    'arrival_mode':  train_meta['arrival_mode'].values,
})

bias_df['undertriage'] = (bias_df['pred_esi'] > bias_df['true_esi']).astype(int)
bias_df['overtriage']  = (bias_df['pred_esi'] < bias_df['true_esi']).astype(int)
bias_df['correct']     = (bias_df['pred_esi'] == bias_df['true_esi']).astype(int)

print('Overall triage accuracy:')
print(f'  Correct    : {bias_df["correct"].mean()*100:.2f}%')
print(f'  Undertriage: {bias_df["undertriage"].mean()*100:.2f}%')
print(f'  Overtriage : {bias_df["overtriage"].mean()*100:.2f}%')

In [ ]:
# 11a — Subgroup analysis by sex
print('\n=== BIAS ANALYSIS BY SEX ===\n')

sex_results = []
for sex in sorted(bias_df['sex'].unique()):
    subset = bias_df[bias_df['sex'] == sex]
    f1  = f1_score(subset['true_esi'], subset['pred_esi'],
                   average='weighted', labels=[1,2,3,4,5])
    ut  = subset['undertriage'].mean() * 100
    ot  = subset['overtriage'].mean() * 100
    acc = subset['correct'].mean() * 100
    sex_results.append({'sex': sex, 'n': len(subset),
                        'accuracy_%': round(acc, 2),
                        'undertriage_%': round(ut, 2),
                        'overtriage_%': round(ot, 2),
                        'weighted_f1': round(f1, 4)})
    print(f'Sex: {sex} (n={len(subset):,})')
    print(f'  Accuracy: {acc:.2f}% | Undertriage: {ut:.2f}% | Overtriage: {ot:.2f}%')
    print(f'  Weighted F1: {f1:.4f}')
    print(classification_report(subset['true_esi'], subset['pred_esi'],
                                 labels=[1,2,3,4,5],
                                 target_names=['ESI 1','ESI 2','ESI 3','ESI 4','ESI 5'],
                                 zero_division=0))

sex_results_df = pd.DataFrame(sex_results)
print(sex_results_df.to_string())

In [ ]:
# 11b — Subgroup analysis by age group
print('\n=== BIAS ANALYSIS BY AGE GROUP ===\n')

age_results = []
for age_grp in sorted(bias_df['age_group'].unique()):
    subset = bias_df[bias_df['age_group'] == age_grp]
    f1  = f1_score(subset['true_esi'], subset['pred_esi'],
                   average='weighted', labels=[1,2,3,4,5])
    ut  = subset['undertriage'].mean() * 100
    ot  = subset['overtriage'].mean() * 100
    acc = subset['correct'].mean() * 100
    age_results.append({'age_group': age_grp, 'n': len(subset),
                        'accuracy_%': round(acc, 2),
                        'undertriage_%': round(ut, 2),
                        'overtriage_%': round(ot, 2),
                        'weighted_f1': round(f1, 4)})
    print(f'Age group: {age_grp} (n={len(subset):,})')
    print(f'  Accuracy: {acc:.2f}% | Undertriage: {ut:.2f}% | Overtriage: {ot:.2f}%')
    print(f'  F1: {f1:.4f}')

age_results_df = pd.DataFrame(age_results)
print(age_results_df.to_string())

In [ ]:
# 11c — Undertriage heatmap: sex × age group
pivot = bias_df.groupby(['sex', 'age_group'])['undertriage'].mean() * 100
pivot = pivot.unstack()

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='Reds', ax=ax,
            cbar_kws={'label': 'Undertriage Rate (%)'}, linewidths=0.5)
ax.set_title('Undertriage Rate (%) by Sex × Age Group', fontsize=13)
ax.set_xlabel('Age Group')
ax.set_ylabel('Sex')
plt.tight_layout()
plt.savefig(WORKING_DIR + 'bias_undertriage_heatmap.png', dpi=150)
plt.show()
print('Undertriage rates (%):\n', pivot.round(2))

In [ ]:
from sklearn.metrics import confusion_matrix

# 11d — Confusion matrices per subgroup
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
subgroups = [('sex', 'F', axes[0]), ('sex', 'M', axes[1]), ('age_group', 'elderly', axes[2])]

for col, val, ax in subgroups:
    subset = bias_df[bias_df[col] == val]
    if len(subset) == 0:
        ax.set_title(f'{col}={val} (no data)'); continue
    cm = confusion_matrix(subset['true_esi'], subset['pred_esi'],
                          labels=[1,2,3,4,5], normalize='true') * 100
    sns.heatmap(cm, annot=True, fmt='.1f', cmap='Blues', ax=ax,
                xticklabels=[1,2,3,4,5], yticklabels=[1,2,3,4,5])
    ax.set_title(f'{col}={val} (n={len(subset):,})')
    ax.set_xlabel('Predicted ESI')
    ax.set_ylabel('True ESI')

plt.suptitle('Confusion Matrices by Subgroup (% of true class)', fontsize=13)
plt.tight_layout()
plt.savefig(WORKING_DIR + 'bias_confusion_matrices.png', dpi=150)
plt.show()

In [ ]:
# 11e — Arrival mode analysis
print('\n=== BIAS ANALYSIS BY ARRIVAL MODE ===\n')

arrival_results = []
for mode in sorted(bias_df['arrival_mode'].unique()):
    subset = bias_df[bias_df['arrival_mode'] == mode]
    f1  = f1_score(subset['true_esi'], subset['pred_esi'],
                   average='weighted', labels=[1,2,3,4,5])
    ut  = subset['undertriage'].mean() * 100
    ot  = subset['overtriage'].mean() * 100
    arrival_results.append({'arrival_mode': mode, 'n': len(subset),
                             'undertriage_%': round(ut, 2),
                             'overtriage_%': round(ot, 2),
                             'weighted_f1': round(f1, 4)})

arrival_results_df = pd.DataFrame(arrival_results)
print(arrival_results_df.sort_values('undertriage_%', ascending=False).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
arrival_results_df.set_index('arrival_mode')[['undertriage_%','overtriage_%']].plot(
    kind='bar', ax=axes[0], color=['tomato','steelblue'], edgecolor='black')
axes[0].set_title('Triage Error Rates by Arrival Mode')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right')
arrival_results_df.set_index('arrival_mode')['weighted_f1'].plot(
    kind='bar', ax=axes[1], color='seagreen', edgecolor='black')
axes[1].set_title('Weighted F1 by Arrival Mode')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.savefig(WORKING_DIR + 'bias_arrival_mode.png', dpi=150)
plt.show()

bias_summary = {
    'overall': {
        'accuracy':         round(bias_df['correct'].mean()*100, 2),
        'undertriage_rate': round(bias_df['undertriage'].mean()*100, 2),
        'overtriage_rate':  round(bias_df['overtriage'].mean()*100, 2)
    },
    'by_sex':          sex_results,
    'by_age_group':    age_results,
    'by_arrival_mode': arrival_results
}
with open(WORKING_DIR + 'bias_analysis_results.json', 'w') as f:
    json.dump(bias_summary, f, indent=2)
print('\nBias analysis results saved.')

## Section 12 — Phase 3: SHAP Explainability

In [ ]:
import shap

shap_model = lgbm_models[-1]  # last fold model
print('Computing SHAP values (takes a few minutes)...')
explainer   = shap.TreeExplainer(shap_model)
np.random.seed(42)
sample_idx  = np.random.choice(len(X_train_full), 5000, replace=False)
X_sample    = X_train_full.iloc[sample_idx]
shap_values = explainer.shap_values(X_sample)

if isinstance(shap_values, list):
    print(f'SHAP values computed. Shape per class: {shap_values[0].shape}')
else:
    print(f'SHAP values computed. Shape: {shap_values.shape}')

In [ ]:
# 12a — Global feature importance (all classes)
if isinstance(shap_values, list):
    mean_shap = np.mean([np.abs(shap_values[c]) for c in range(5)], axis=0)
else:
    mean_shap = np.mean(np.abs(shap_values), axis=2)

global_importance = pd.DataFrame({
    'feature':       X_train_full.columns,
    'mean_abs_shap': mean_shap.mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)

print('Top 20 features by mean |SHAP|:')
print(global_importance.head(20).to_string())

fig, ax = plt.subplots(figsize=(10, 8))
top20 = global_importance.head(20)
ax.barh(top20['feature'][::-1], top20['mean_abs_shap'][::-1], color='steelblue')
ax.set_title('Top 20 Features — Mean |SHAP| (All Classes)', fontsize=13)
ax.set_xlabel('Mean |SHAP Value|')
plt.tight_layout()
plt.savefig(WORKING_DIR + 'shap_global_top20.png', dpi=150)
plt.show()

In [ ]:
# 12b — SHAP bar + beeswarm for ESI 1
def get_class_shap(shap_values, class_idx):
    if isinstance(shap_values, list):
        return shap_values[class_idx]
    return shap_values[:, :, class_idx]

fig, ax = plt.subplots(figsize=(12, 8))
shap.summary_plot(get_class_shap(shap_values, 0), X_sample,
                  max_display=20, show=False, plot_type='bar')
plt.title('SHAP Feature Importance — ESI 1 (Most Critical)', fontsize=13)
plt.tight_layout()
plt.savefig(WORKING_DIR + 'shap_esi1_bar.png', dpi=150, bbox_inches='tight')
plt.show()

fig = plt.figure(figsize=(12, 8))
shap.summary_plot(get_class_shap(shap_values, 0), X_sample,
                  max_display=20, show=False)
plt.title('SHAP Beeswarm — ESI 1', fontsize=13)
plt.tight_layout()
plt.savefig(WORKING_DIR + 'shap_esi1_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 12c — Waterfall plots for representative ESI 1, 3, 5 patients
sample_labels = y[sample_idx]

esi1_idx = np.where(sample_labels == 0)[0][0]
esi3_idx = np.where(sample_labels == 2)[0][0]
esi5_idx = np.where(sample_labels == 4)[0][0]

for esi_label, idx, class_idx in [
    ('ESI 1', esi1_idx, 0),
    ('ESI 3', esi3_idx, 2),
    ('ESI 5', esi5_idx, 4)
]:
    print(f'\nWaterfall plot for {esi_label} patient...')
    shap_exp = shap.Explanation(
        values=get_class_shap(shap_values, class_idx)[idx],
        base_values=explainer.expected_value[class_idx],
        data=X_sample.iloc[idx].values,
        feature_names=X_train_full.columns.tolist()
    )
    fig = plt.figure(figsize=(12, 6))
    shap.waterfall_plot(shap_exp, max_display=15, show=False)
    plt.title(f'SHAP Waterfall — Representative {esi_label} Patient')
    plt.tight_layout()
    plt.savefig(WORKING_DIR + f'shap_waterfall_{esi_label.replace(" ","")}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

## Section 13 — Final Submission & Output Verification

In [ ]:
# Generate submission CSV
test_raw_sub = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

final_test_labels = np.argmax(final_test_preds, axis=1) + 1
submission = pd.DataFrame({
    'patient_id':    test_raw_sub['patient_id'],
    'triage_acuity': final_test_labels
})

print('Submission shape:', submission.shape)
print('\nPredicted ESI distribution:')
print(submission['triage_acuity'].value_counts().sort_index())
print('\nExpected (from training):')
print(pd.Series(y + 1).value_counts().sort_index())

assert submission.isnull().sum().sum() == 0, 'NaN in submission!'
assert submission['triage_acuity'].between(1, 5).all(), 'ESI out of range!'
assert len(submission) == len(test_raw_sub), 'Row count mismatch!'

submission.to_csv(WORKING_DIR + 'submission_final.csv', index=False)
print('\nsubmission_final.csv saved.')
print(submission.head())

In [ ]:
# Final summary + output file check
final_summary = {
    'baseline_f1':         round(oof_f1, 4),
    'baseline_qwk':        round(oof_kappa, 4),
    'phase2_f1':           round(oof_f1_v2, 4),
    'phase2_qwk':          round(oof_kappa_v2, 4),
    'optuna_best_f1':      round(study.best_value, 4),
    'lgbm_tuned_f1':       round(lgbm_f1, 4),
    'lgbm_tuned_qwk':      round(lgbm_kappa, 4),
    'xgb_f1':              round(xgb_f1, 4),
    'xgb_qwk':             round(xgb_kappa, 4),
    'ensemble_f1':         round(final_f1, 4),
    'ensemble_qwk':        round(final_kappa, 4),
    'ensemble_method':     'lgbm_xgb_soft_blend',
    'blend_weights':       {'lgbm': round(best_w[0], 2), 'xgb': round(best_w[1], 2)},
    'nlp_uplift_f1':       round(oof_f1_v2 - oof_f1, 4),
    'bias_undertriage':    bias_summary['overall']['undertriage_rate'],
    'total_features':      X_train_full.shape[1],
}

with open(WORKING_DIR + 'final_summary.json', 'w') as f:
    json.dump(final_summary, f, indent=2)

print('Final summary:')
print(json.dumps(final_summary, indent=2))

print('\n=== Output File Check ===')
required = [
    'oof_baseline.npy', 'test_baseline.npy',
    'train_bert.npy', 'test_bert.npy',
    'pca_scree.png',
    'X_train_full.parquet', 'X_test_full.parquet',
    'oof_phase2.npy', 'test_phase2.npy',
    'oof_lgbm_tuned.npy', 'test_lgbm_tuned.npy',
    'oof_xgb.npy', 'test_xgb.npy',
    'final_test_preds.npy', 'final_oof_preds.npy',
    'bias_analysis_results.json',
    'bias_undertriage_heatmap.png',
    'bias_confusion_matrices.png',
    'bias_arrival_mode.png',
    'shap_global_top20.png',
    'shap_esi1_bar.png',
    'shap_esi1_beeswarm.png',
    'shap_waterfall_ESI1.png',
    'shap_waterfall_ESI3.png',
    'shap_waterfall_ESI5.png',
    'submission_final.csv',
    'final_summary.json',
]
all_ok = True
for fname in required:
    exists = os.path.exists(WORKING_DIR + fname)
    status = '✅' if exists else '❌ MISSING'
    print(f'  {status}  {fname}')
    if not exists:
        all_ok = False

print()
print('All files present.' if all_ok else '⚠️  Some files missing — check above.')